# Notebook: Fine-tune OPUS-MT vi-en (System B và các ablation)

Fine-tune `Helsinki-NLP/opus-mt-vi-en` trên cặp (câu nhận dạng của PhoWhisper → bản dịch VinAI) và các lần chạy đối chứng. Mỗi lần chạy **độc lập, bắt đầu từ mô hình tiền huấn luyện**, và làm tiếp được qua nhiều phiên Colab.

| Lần chạy | Nguồn (đầu vào) | Số mẫu train | Ý nghĩa |
|---|---|---|---|
| `B_100` | `asr_text` | 49,500 | System B đầy đủ |
| `clean_100` | `vi_text` | 49,500 | Đối chứng: huấn luyện trên transcript sạch |
| `B_50` | `asr_text` | 24,750 | Ablation kích thước dữ liệu |
| `B_25` | `asr_text` | 12,375 | Ablation kích thước dữ liệu |

- **Đích** (nhãn) luôn là `en_text` (bản dịch VinAI Translate v2, đã sửa lỗi tách `n't`).
- **Dev** (5,500 mẫu) luôn có nguồn là `asr_text`, cho cả bốn lần chạy, vì lúc test đầu vào luôn là câu ASR. Early stopping dựa trên dev loss.
- Các tập con train là **tiền tố** theo `pos` (vị trí trong hoán vị ngẫu nhiên cố định của notebook ASR), nên lồng nhau: 25% ⊂ 50% ⊂ 100%.
- Dòng có lỗi đọc audio hoặc ASR rỗng bị loại khỏi cả train lẫn dev (số lượng được in ra).

**Cách chạy:** chạy lần lượt các ô. Đặt `SMOKE = True` trước để thử nhanh toàn bộ đường ống (vài phút, ghi vào thư mục riêng), rồi đặt lại `False`. Phiên bị ngắt thì chạy lại từ ô 1: lần chạy đã xong được bỏ qua, lần chạy dở làm tiếp từ checkpoint gần nhất.

In [11]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [12]:
!pip -q install -U transformers accelerate sentencepiece sacremoses

In [13]:
import os, re, json, time, shutil, random
import numpy as np, pandas as pd, torch

# ---------- ĐƯỜNG DẪN ----------
ROOT        = '/content/drive/MyDrive/project_DPL'
ASR_HYP_CSV = os.path.join(ROOT, 'results/asr_out/asr_hyp.csv')                          # từ notebook ASR (ô 4)
META_CSV    = os.path.join(ROOT, 'datasets/vietspeech_100k/metadata_with_en_vinai.csv')  # nguồn của en_text
MODEL_DIR   = os.path.join(ROOT, 'models/opus-mt-vi-en')                                 # chưa có thì tải một lần
MT_MODEL    = 'Helsinki-NLP/opus-mt-vi-en'
FT_DIR      = os.path.join(ROOT, 'results/ft_out')                                       # kết quả fine-tune (PHẢI nằm trên Drive)

# ---------- DỮ LIỆU (khớp notebook ASR) ----------
SEED, N_DEV, N_TRAIN = 42, 5500, 49500

# ---------- SIÊU THAM SỐ (đã chốt) ----------
MAX_LEN, BATCH, EVAL_BATCH = 128, 64, 128
LR, WEIGHT_DECAY, WARMUP_FRAC = 2e-5, 0.01, 0.05
MAX_EPOCHS, PATIENCE = 30, 5          # early stopping theo dev loss
FP16 = True                           # chỉ có tác dụng khi có GPU; nếu loss bất thường (nan) thì đặt False
KEEP_CKPT = 2                         # số checkpoint giữ lại mỗi lần chạy (mỗi cái khoảng 0.9 GB: mô hình + trạng thái optimizer)
DELETE_CKPT_AFTER = True              # xóa checkpoint khi lần chạy xong (giữ lại mô hình tốt nhất)
TIME_LIMIT_H = None                   # None = chạy đến hết; hoặc số giờ để dừng êm ở cuối một epoch (chạy lại để làm tiếp)

RUNS = [                              # theo thứ tự ưu tiên
    dict(name='B_100',     src='asr_text', frac=1.00),
    dict(name='clean_100', src='vi_text',  frac=1.00),
    dict(name='B_50',      src='asr_text', frac=0.50),
    dict(name='B_25',      src='asr_text', frac=0.25),
]

SMOKE = False                         # True: thử nhanh đường ống (500 mẫu train, 200 dev, 1 epoch, ghi vào ft_out_smoke)
if SMOKE:
    FT_DIR = FT_DIR + '_smoke'; MAX_EPOCHS = 1; SMOKE_TRAIN, SMOKE_DEV = 500, 200

os.makedirs(FT_DIR, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '|', torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU (rất chậm)', '| SMOKE =', SMOKE)

device: cuda | Tesla T4 | SMOKE = False


In [14]:
# 1) Dữ liệu: asr_hyp.csv + en_text lấy từ metadata theo id
hyp = pd.read_csv(ASR_HYP_CSV, keep_default_na=False)
for c in ['id', 'vi_text', 'asr_text', 'error', 'pos', 'split']: assert c in hyp.columns, f'Thiếu cột {c} trong asr_hyp.csv'
assert hyp.id.is_unique
assert len(hyp) == N_DEV + N_TRAIN, (f'asr_hyp.csv có {len(hyp):,} dòng, cần {N_DEV + N_TRAIN:,}. '
    'Hãy hoàn tất notebook ASR rồi chạy lại ô gộp (ô 4) để tạo asr_hyp.csv đầy đủ.')

meta = pd.read_csv(META_CSV, usecols=['id', 'en_text'], keep_default_na=False)
en = dict(zip(meta.id, meta.en_text))
assert all(i in en for i in hyp.id), 'Có id trong asr_hyp.csv không có trong metadata'
hyp['en_text'] = hyp.id.map(en).astype(str)          # bỏ bản sao en_text cũ trong asr_hyp.csv

# sửa lỗi tách n't của VinAI: "do n't" -> "don't" (các dạng khác như 's, 'll vẫn đúng)
NT = re.compile(r"\s+(n['’]t)\b")
before = hyp.en_text.copy(); hyp['en_text'] = before.map(lambda s: NT.sub(r'\1', s))
chg = hyp.en_text != before
print(f"Đã sửa n't: {int(chg.sum()):,}/{len(hyp):,} câu")
for b, a in list(zip(before[chg], hyp.en_text[chg]))[:3]: print('   ', b, '->', a)

# loại dòng lỗi
ok = (hyp.error.astype(str).str.strip() == '') & (hyp.asr_text.astype(str).str.strip() != '') & (hyp.en_text.str.strip() != '') & (hyp.vi_text.astype(str).str.strip() != '')
print(f'Loại {int((~ok).sum()):,} dòng (lỗi đọc audio / ASR rỗng / trống): dev {int(((~ok) & (hyp.split == "dev")).sum())}, train {int(((~ok) & (hyp.split == "train")).sum())}')

dev = hyp[ok & (hyp.split == 'dev')].sort_values('pos')
def train_subset(frac):
    n = int(round(frac * N_TRAIN))                    # tiền tố theo vị trí; lọc dòng lỗi sau khi cắt, nên các tập vẫn lồng nhau
    return hyp[ok & (hyp.split == 'train') & (hyp.pos < N_DEV + n)].sort_values('pos')
if SMOKE:
    dev = dev.head(SMOKE_DEV)
    _ts = train_subset; train_subset = lambda frac: _ts(frac).head(SMOKE_TRAIN)
print('dev:', len(dev), '| train 25/50/100%:', [len(train_subset(f)) for f in (0.25, 0.5, 1.0)])
print('\nVí dụ (nguồn ASR | nguồn sạch | đích):')
for _, r in train_subset(1.0).head(3).iterrows(): print(' -', r.asr_text, '|', r.vi_text, '|', r.en_text)

Đã sửa n't: 4,095/55,000 câu
    Well, do n't hit too much. Yeah, that's right. -> Well, don't hit too much. Yeah, that's right.
    We figured it out, but we did n't say it. Slowly, slowly, finally. And now I'm inviting two people to be called... -> We figured it out, but we didn't say it. Slowly, slowly, finally. And now I'm inviting two people to be called...
    But I do n't do that because not everyone understands pure love like I do. -> But I don't do that because not everyone understands pure love like I do.
Loại 0 dòng (lỗi đọc audio / ASR rỗng / trống): dev 0, train 0
dev: 5500 | train 25/50/100%: [12375, 24750, 49500]

Ví dụ (nguồn ASR | nguồn sạch | đích):
 - có thể sẽ rất khó khăn. | có thể sẽ rất khó khăn | can be very difficult
 - vùng có ít nhất sáu trăm hai mươi ngàn người được cho là thuộc dân tộc hán trên tổng thể ước chừng tấn cương có khoảng bốn mươi phần trăm. | có ít nhất sáu trăm hai mươi nghìn người được cho là thuộc dân tộc hán trên tổng thể ước chừng tân cương

In [15]:
# 2) Mô hình và tokenizer (tải một lần về MODEL_DIR trên Drive)
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments,
                          DataCollatorForSeq2Seq, EarlyStoppingCallback, TrainerCallback)
from transformers.trainer_utils import get_last_checkpoint

if not os.path.isfile(os.path.join(MODEL_DIR, 'config.json')):
    _tmp = MODEL_DIR + '.tmp'; shutil.rmtree(_tmp, ignore_errors=True); os.makedirs(os.path.dirname(MODEL_DIR), exist_ok=True)
    _t = AutoTokenizer.from_pretrained(MT_MODEL); _m = AutoModelForSeq2SeqLM.from_pretrained(MT_MODEL)
    _t.save_pretrained(_tmp); _m.save_pretrained(_tmp)
    assert os.path.isfile(os.path.join(_tmp, 'config.json')); shutil.rmtree(MODEL_DIR, ignore_errors=True); os.rename(_tmp, MODEL_DIR)
    del _t, _m; print('Đã lưu mô hình vào', MODEL_DIR)

tok = AutoTokenizer.from_pretrained(MODEL_DIR)
n_params = sum(p.numel() for p in AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR).parameters())
print(f'Nạp từ {MODEL_DIR} | {n_params/1e6:.1f} triệu tham số')

class PairDS(torch.utils.data.Dataset):
    def __init__(self, src, tgt):
        src, tgt = list(map(str, src)), list(map(str, tgt)); self.ids, self.mask, self.lab = [], [], []
        for i in range(0, len(src), 5000):
            e = tok(src[i:i+5000], text_target=tgt[i:i+5000], max_length=MAX_LEN, truncation=True)
            self.ids += e['input_ids']; self.mask += e['attention_mask']; self.lab += e['labels']
    def __len__(self): return len(self.ids)
    def __getitem__(self, i): return {'input_ids': self.ids[i], 'attention_mask': self.mask[i], 'labels': self.lab[i]}

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Nạp từ /content/drive/MyDrive/project_DPL/models/opus-mt-vi-en | 72.2 triệu tham số


In [16]:
# 3) Hàm huấn luyện một lần chạy (làm tiếp được; bỏ qua nếu đã xong)
class TimingCB(TrainerCallback):
    # ghi tổng thời gian huấn luyện cộng dồn qua các phiên và VRAM đỉnh
    def __init__(self, path):
        self.path = path
        prev = json.load(open(path)) if os.path.exists(path) else {}
        self.prev_s, self.prev_peak = prev.get('train_s', 0.0), prev.get('peak_vram_gb', 0.0); self.t0 = None
    def _flush(self):
        peak = max(self.prev_peak, torch.cuda.max_memory_allocated() / 2**30 if torch.cuda.is_available() else 0.0)
        json.dump({'train_s': self.prev_s + time.time() - self.t0, 'peak_vram_gb': peak}, open(self.path, 'w'))
    def on_train_begin(self, args, state, control, **kw): self.t0 = time.time()
    def on_epoch_end(self, args, state, control, **kw): self._flush()
    def on_train_end(self, args, state, control, **kw): self._flush()

class TimeLimitCB(TrainerCallback):
    def __init__(self, deadline): self.deadline, self.hit = deadline, False
    def on_epoch_end(self, args, state, control, **kw):
        if self.deadline is not None and time.time() > self.deadline: self.hit = True; control.should_training_stop = True

T_DEADLINE = None if TIME_LIMIT_H is None else time.time() + TIME_LIMIT_H * 3600

def run_ft(name, src_col, frac):
    out = os.path.join(FT_DIR, name); ck = os.path.join(out, 'ckpt'); best = os.path.join(out, 'best_model')
    st_path, tm_path = os.path.join(out, 'status.json'), os.path.join(out, 'timing.json')
    if os.path.exists(st_path) and os.path.isfile(os.path.join(best, 'config.json')) and json.load(open(st_path)).get('done'):
        print(f'[{name}] đã xong, bỏ qua.'); return json.load(open(st_path))
    os.makedirs(out, exist_ok=True)
    tr = train_subset(frac)
    train_ds, dev_ds = PairDS(tr[src_col], tr.en_text), PairDS(dev.asr_text, dev.en_text)   # dev luôn nguồn ASR
    steps_ep = -(-len(train_ds) // BATCH); warm = int(np.ceil(WARMUP_FRAC * steps_ep * MAX_EPOCHS))
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR)
    args = Seq2SeqTrainingArguments(
        output_dir=ck, num_train_epochs=MAX_EPOCHS, per_device_train_batch_size=BATCH, per_device_eval_batch_size=EVAL_BATCH,
        learning_rate=LR, weight_decay=WEIGHT_DECAY, warmup_steps=warm, lr_scheduler_type='linear',
        eval_strategy='epoch', save_strategy='epoch', save_total_limit=KEEP_CKPT, logging_strategy='epoch',
        load_best_model_at_end=True, metric_for_best_model='eval_loss', greater_is_better=False,
        fp16=(FP16 and DEVICE == 'cuda'), seed=SEED, data_seed=SEED, report_to='none', dataloader_num_workers=2,
        predict_with_generate=False)
    tl = TimeLimitCB(T_DEADLINE)
    trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=dev_ds,
        data_collator=DataCollatorForSeq2Seq(tok, model=model, label_pad_token_id=-100), processing_class=tok,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE), TimingCB(tm_path), tl])
    last = get_last_checkpoint(ck) if os.path.isdir(ck) else None
    print(f'[{name}] nguồn={src_col} | train {len(train_ds):,} | dev {len(dev_ds):,} | {steps_ep} bước/epoch | warmup {warm} bước | ' + (f'làm tiếp từ {os.path.basename(last)}' if last else 'bắt đầu mới'))
    if DEVICE == 'cuda': torch.cuda.reset_peak_memory_stats()
    trainer.train(resume_from_checkpoint=last)
    if tl.hit:
        print(f'[{name}] hết TIME_LIMIT_H; dừng ở epoch {trainer.state.epoch:.0f}. Chạy lại để làm tiếp.'); return None
    trainer.save_model(best); tok.save_pretrained(best)                       # mô hình tốt nhất (đã nạp lại nhờ load_best_model_at_end)
    json.dump(trainer.state.log_history, open(os.path.join(out, 'log_history.json'), 'w'))
    tm = json.load(open(tm_path))
    res = dict(name=name, src=src_col, frac=frac, n_train=len(train_ds), n_dev=len(dev_ds), done=True,
               epochs_run=round(float(trainer.state.epoch), 2), stopped_early=bool(trainer.state.epoch < MAX_EPOCHS - 1e-6),
               best_dev_loss=float(trainer.state.best_metric), train_s=round(tm['train_s'], 1), peak_vram_gb=round(tm['peak_vram_gb'], 2),
               batch=BATCH, lr=LR, fp16=bool(args.fp16), seed=SEED)
    json.dump(res, open(st_path, 'w'), indent=1)
    if DELETE_CKPT_AFTER: shutil.rmtree(ck, ignore_errors=True)
    print(f'[{name}] XONG: {res["epochs_run"]} epoch, dev loss tốt nhất {res["best_dev_loss"]:.4f}, {res["train_s"]/60:.1f} phút, VRAM đỉnh {res["peak_vram_gb"]} GB')
    del trainer, model; torch.cuda.empty_cache()
    return res

In [17]:
# 4) Chạy lần lượt các lần chạy (làm tiếp được)
results = []
for r in RUNS:
    if T_DEADLINE is not None and time.time() > T_DEADLINE: print('Hết TIME_LIMIT_H; dừng.'); break
    out = run_ft(r['name'], r['src'], r['frac'])
    if out is None: break
    results.append(out)

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

[B_100] nguồn=asr_text | train 49,500 | dev 5,500 | 774 bước/epoch | warmup 1161 bước | bắt đầu mới


Epoch,Training Loss,Validation Loss
1,1.855664,1.616328
2,1.597358,1.523044
3,1.418034,1.486633
4,1.283389,1.471755
5,1.171863,1.465051
6,1.080387,1.464451


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.encoder.embed_positions.weight', 'model.decoder.embed_tokens.weight', 'model.decoder.embed_positions.weight', 'lm_head.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[B_25] XONG: 9.0 epoch, dev loss tốt nhất 1.6063, 6.2 phút, VRAM đỉnh 7.84 GB


In [18]:
# 5) Bảng tóm tắt các lần chạy đã xong -> runs_summary.csv
rows = []
for r in RUNS:
    p = os.path.join(FT_DIR, r['name'], 'status.json')
    if os.path.exists(p): rows.append(json.load(open(p)))
summary = pd.DataFrame(rows)
if len(summary):
    summary['train_min'] = (summary.train_s / 60).round(1)
    tmp = os.path.join(FT_DIR, 'runs_summary.csv.tmp'); summary.to_csv(tmp, index=False); os.replace(tmp, os.path.join(FT_DIR, 'runs_summary.csv'))
    print(summary[['name', 'src', 'n_train', 'epochs_run', 'stopped_early', 'best_dev_loss', 'train_min', 'peak_vram_gb']].to_string(index=False))
else:
    print('Chưa có lần chạy nào xong.')
print(f'\nCòn lại: {[r["name"] for r in RUNS if not os.path.exists(os.path.join(FT_DIR, r["name"], "status.json"))]}')

     name      src  n_train  epochs_run  stopped_early  best_dev_loss  train_min  peak_vram_gb
    B_100 asr_text    49500        11.0           True       1.464451       23.8          7.84
clean_100  vi_text    49500         9.0           True       1.599896       19.7          7.83
     B_50 asr_text    24750        10.0           True       1.546614       12.5          7.83
     B_25 asr_text    12375         9.0           True       1.606327        6.2          7.84

Còn lại: []


In [21]:
# 6) Xem thử: dịch vài câu dev bằng mô hình tốt nhất của lần chạy đầu tiên đã xong
done_runs = [r['name'] for r in RUNS if os.path.exists(os.path.join(FT_DIR, r['name'], 'best_model', 'config.json'))]
if done_runs:
    bm = os.path.join(FT_DIR, done_runs[0], 'best_model')
    t2, m2 = AutoTokenizer.from_pretrained(bm), AutoModelForSeq2SeqLM.from_pretrained(bm).to(DEVICE).eval()
    smp = dev.sample(5, random_state=42)
    with torch.no_grad(): o = m2.generate(**t2(list(smp.asr_text), return_tensors='pt', padding=True, truncation=True, max_length=MAX_LEN).to(DEVICE), num_beams=4, max_new_tokens=MAX_LEN)
    print(f'Mô hình: {done_runs[0]}')
    for (_, r), h in zip(smp.iterrows(), t2.batch_decode(o, skip_special_tokens=True)): print(f' ASR: {r.asr_text}\n REF: {r.en_text}\n HYP: {h}\n')

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=128) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Mô hình: B_100
 ASR: một ngày mà mình gặp nhau tới hai lần lúc đó trong đầu anh nghĩ là.
 REF: The day we met twice, I was thinking,
 HYP: One day we met twice, and then in my head, I thought...

 ASR: cục hát việt nam.
 REF: Vietnamese songs
 HYP: Vietnamese song club

 ASR: thái độ bức thuốc thẳng thắn của ông đã khiến cho các tu diện lúc đó ngần ngại không dám chứa chấp ông.
 REF: His outspoken indomitable attitude made the monasteries at the time reluctant to harbor him.
 HYP: His straightforward attitude made the monks hesitate to accept him.

 ASR: quân đoán và không thường thấy ở cầu thủ này.
 REF: assertive and not often seen in this player
 HYP: The army guesses and doesn't usually see it in this player.

 ASR: kể từ khi cải cách mở cửa và gia nhập về kpto.
 REF: Since reforms opened and joined the WTO, China has risen
 HYP: since reform opened the door and joined KPto



In [20]:
# 7) (Chỉ trên Colab) Đẩy toàn bộ phần ghi lên Drive trước khi kết thúc phiên
try:
    from google.colab import drive
    drive.flush_and_unmount(); print('Đã flush Drive.')
except Exception as e:
    print('Bỏ qua flush:', e)